# 1. Carga de datos

In [1]:
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

PROJECT_ROOT = os.path.abspath(os.getcwd())
if os.path.basename(PROJECT_ROOT).lower() == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
IMAGES_DIR = os.path.join(PROJECT_ROOT, "images")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
rows = []
for class_name in sorted(os.listdir(DATA_DIR)):
    class_dir = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_dir) or class_name.startswith("."):
        continue
    for file_name in sorted(os.listdir(class_dir)):
        if file_name.startswith("."):
            continue
        if os.path.splitext(file_name)[1].lower() in valid_exts:
            rows.append({"ruta": os.path.join(class_dir, file_name), "clase": class_name})

data_df = pd.DataFrame(rows)
class_names = sorted(data_df["clase"].unique().tolist())
print("Total de imágenes:", len(data_df))
print("Clases:", class_names)
display(data_df.head())

Total de imágenes: 2000
Clases: ['Tomato___Bacterial_spot', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___healthy']


,ruta,clase
0,C:\Users\sj-alumno\Downloads\PlantVillageTomat...,Tomato___Bacterial_spot
1,C:\Users\sj-alumno\Downloads\PlantVillageTomat...,Tomato___Bacterial_spot
2,C:\Users\sj-alumno\Downloads\PlantVillageTomat...,Tomato___Bacterial_spot
3,C:\Users\sj-alumno\Downloads\PlantVillageTomat...,Tomato___Bacterial_spot
4,C:\Users\sj-alumno\Downloads\PlantVillageTomat...,Tomato___Bacterial_spot


# 2. EDA (Análisis Exploratorio de Datos)

In [2]:
counts = data_df["clase"].value_counts().sort_index()
percentages = counts / counts.sum() * 100

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index, legend=False)
ax.set_title("Distribución de imágenes por clase")
ax.set_xlabel("Clase")
ax.set_ylabel("Número de imágenes")
plt.xticks(rotation=35, ha="right")
for i, (count, pct) in enumerate(zip(counts.values, percentages.values)):
    ax.text(i, count, f"{count}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "eda_distribucion_clases.png"), dpi=150, bbox_inches="tight")
plt.close()

resolution_rows = []
rgb_samples = []
for _, row in data_df.iterrows():
    try:
        with Image.open(row["ruta"]) as pil_img:
            rgb = pil_img.convert("RGB")
            width, height = rgb.size
            channels = len(rgb.getbands())
            resolution_rows.append({"ancho": width, "alto": height, "canales": channels})
            if len(rgb_samples) < 500:
                arr = np.asarray(rgb, dtype=np.uint8)
                rgb_samples.append(arr)
    except (OSError, ValueError):
        continue

resolution_df = pd.DataFrame(resolution_rows)
print("Resolución original más frecuente:")
print(resolution_df.groupby(["ancho", "alto", "canales"]).size().sort_values(ascending=False).head())

if rgb_samples:
    rgb_pixels = np.concatenate([arr.reshape(-1, 3) for arr in rgb_samples], axis=0)
    plt.figure(figsize=(12, 4))
    colors = ["red", "green", "blue"]
    for i, color in enumerate(colors):
        plt.hist(rgb_pixels[:, i], bins=64, alpha=0.55, color=color, label=f"{color.upper()}")
    plt.title("Distribución de valores RGB")
    plt.xlabel("Valor de píxel")
    plt.ylabel("Frecuencia")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "eda_distribucion_rgb.png"), dpi=150, bbox_inches="tight")
    plt.close()

n_classes = len(class_names)
plt.figure(figsize=(15, max(4, 3 * ((n_classes + 3) // 4))))
for i, class_name in enumerate(class_names):
    sample_row = data_df[data_df["clase"] == class_name].iloc[0]
    with Image.open(sample_row["ruta"]) as pil_img:
        img = pil_img.convert("RGB")
        plt.subplot((n_classes + 3) // 4, 4, i + 1)
        plt.imshow(img)
        plt.title(class_name)
        plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "eda_muestras_clases.png"), dpi=150, bbox_inches="tight")
plt.close()

print("EDA completado.")

Resolución original más frecuente:
ancho  alto  canales
256    256   3          2000
dtype: int64


EDA completado.


# 3. Pre procesamiento

In [3]:
class_to_idx = {name: i for i, name in enumerate(class_names)}
X_list, y_list = [], []
for _, row in data_df.iterrows():
    try:
        with Image.open(row["ruta"]) as pil_img:
            pil_img = pil_img.convert("RGB").resize((128, 128), Image.Resampling.BILINEAR)
            arr = np.asarray(pil_img, dtype=np.float32) / 255.0
            X_list.append(arr)
            y_list.append(class_to_idx[row["clase"]])
    except (OSError, ValueError):
        continue

X = np.asarray(X_list, dtype=np.float32)
y = np.asarray(y_list, dtype=np.int32)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85,
    random_state=SEED, stratify=y_trainval
)

num_classes = len(class_names)
input_shape = (128, 128, 3)
batch_size = 32

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("Rango:", float(X.min()), "-", float(X.max()))
print("Distribución train:", np.bincount(y_train, minlength=num_classes).tolist())


def augment_batch(batch, rng):
    augmented = batch.copy()
    for i in range(len(augmented)):
        if rng.random() < 0.5:
            augmented[i] = np.flip(augmented[i], axis=1)
        if rng.random() < 0.25:
            augmented[i] = np.flip(augmented[i], axis=0)
    tensor = tf.convert_to_tensor(augmented, dtype=tf.float32)
    tensor = tf.image.random_brightness(tensor, max_delta=0.12,
                                        seed=int(rng.integers(0, 2**31 - 1)))
    return tf.clip_by_value(tensor, 0.0, 1.0).numpy().astype(np.float32)

X_train: (1400, 128, 128, 3) float32
X_val:   (300, 128, 128, 3) float32
X_test:  (300, 128, 128, 3) float32
Rango: 0.0 - 1.0
Distribución train: [350, 350, 350, 350]


# 4. Modelo

In [4]:

def build_mlp(input_shape, num_classes, lr=1e-3):
    model = keras.Sequential([
        keras.layers.Input(shape=input_shape),
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(num_classes, activation="softmax"),
    ], name="MLP_Baseline")
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
    return model


def build_cnn_base(input_shape, num_classes, lr=1e-3):
    inputs = keras.layers.Input(shape=input_shape)
    x = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="CNN_Base")
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
    return model

mlp_base = build_mlp(input_shape, num_classes)
cnn_base = build_cnn_base(input_shape, num_classes)
print("MLP Baseline")
mlp_base.summary()
print("\nCNN Base")
cnn_base.summary()


MLP Baseline


Model: "MLP_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 49152)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    25,166,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,298,692 (96.51 MB)

 Trainable params: 25,298,692 (96.51 MB)

 Non-trainable params: 0 (0.00 B)

... (2 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,276 (430.77 KB)

 Trainable params: 110,276 (430.77 KB)

 Non-trainable params: 0 (0.00 B)

# 5. Entrenamiento

In [5]:
def train_custom(model, epochs=15, use_aug=False, seed_offset=0):
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    started = time.time()
    for epoch in range(1, epochs + 1):
        epoch_started = time.time()
        rng = np.random.default_rng(SEED + seed_offset + epoch)
        indices = rng.permutation(len(X_train))
        losses, accuracies = [], []
        for start_i in range(0, len(indices), batch_size):
            batch_idx = indices[start_i:start_i + batch_size]
            batch_x = X_train[batch_idx]
            if use_aug:
                batch_x = augment_batch(batch_x, rng)
            metrics = model.train_on_batch(batch_x, y_train[batch_idx], return_dict=True)
            losses.append(float(metrics["loss"]))
            accuracies.append(float(metrics["accuracy"]))

        val_losses, val_accuracies = [], []
        for start_i in range(0, len(X_val), batch_size):
            metrics = model.test_on_batch(
                X_val[start_i:start_i + batch_size], y_val[start_i:start_i + batch_size], return_dict=True
            )
            val_losses.append(float(metrics["loss"]))
            val_accuracies.append(float(metrics["accuracy"]))

        epoch_values = {
            "loss": float(np.mean(losses)), "accuracy": float(np.mean(accuracies)),
            "val_loss": float(np.mean(val_losses)), "val_accuracy": float(np.mean(val_accuracies)),
        }
        for key, value in epoch_values.items():
            history[key].append(value)
        print(f"Epoch {epoch:02d}/{epochs} - loss={epoch_values['loss']:.4f} - accuracy={epoch_values['accuracy']:.4f} - val_loss={epoch_values['val_loss']:.4f} - val_accuracy={epoch_values['val_accuracy']:.4f} - tiempo={time.time()-epoch_started:.2f}s")
    return history, time.time() - started


def evaluate_and_predict(model):
    probabilities = model.predict(X_test, batch_size=batch_size, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    accuracy = accuracy_score(y_test, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions, average="macro", zero_division=0
    )
    return accuracy, precision, recall, f1, predictions, probabilities

print("Entrenando MLP Base")
mlp_history, mlp_time = train_custom(mlp_base, epochs=15, use_aug=False, seed_offset=100)
mlp_test = evaluate_and_predict(mlp_base)

print("\nEntrenando CNN Base")
cnn_history, cnn_time = train_custom(cnn_base, epochs=15, use_aug=False, seed_offset=200)
cnn_test = evaluate_and_predict(cnn_base)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for name, history in [("MLP Base", mlp_history), ("CNN Base", cnn_history)]:
    plt.plot(history["loss"], label=f"{name} train")
    plt.plot(history["val_loss"], linestyle="--", label=f"{name} val")
plt.title("Loss / Val Loss")
plt.xlabel("Época")
plt.legend()

plt.subplot(1, 2, 2)
for name, history in [("MLP Base", mlp_history), ("CNN Base", cnn_history)]:
    plt.plot(history["accuracy"], label=f"{name} train")
    plt.plot(history["val_accuracy"], linestyle="--", label=f"{name} val")
plt.title("Accuracy / Val Accuracy")
plt.xlabel("Época")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "curvas_entrenamiento_base.png"), dpi=150, bbox_inches="tight")
plt.close()

for title, predictions, filename in [
    ("MLP Base", mlp_test[4], "cm_mlp_base.png"),
    ("CNN Base", cnn_test[4], "cm_cnn_base.png"),
]:
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Matriz de confusión — {title}")
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, filename), dpi=150, bbox_inches="tight")
    plt.close()

print("MLP Base accuracy:", round(mlp_test[0], 4))
print("CNN Base accuracy:", round(cnn_test[0], 4))

Entrenando MLP Base


Epoch 01/15 - loss=11.7926 - accuracy=0.3161 - val_loss=1.3849 - val_accuracy=0.5781 - tiempo=5.33s


Epoch 02/15 - loss=1.3515 - accuracy=0.5698 - val_loss=0.8444 - val_accuracy=0.6896 - tiempo=4.53s


Epoch 03/15 - loss=0.9986 - accuracy=0.6463 - val_loss=0.7422 - val_accuracy=0.7396 - tiempo=4.53s


Epoch 04/15 - loss=0.8711 - accuracy=0.6790 - val_loss=0.7129 - val_accuracy=0.7292 - tiempo=4.54s


Epoch 05/15 - loss=0.7362 - accuracy=0.7313 - val_loss=0.8946 - val_accuracy=0.6594 - tiempo=4.58s


... (23 outputs left out)

Epoch 13/15 - loss=0.2631 - accuracy=0.9186 - val_loss=0.3121 - val_accuracy=0.9031 - tiempo=3.09s


Epoch 14/15 - loss=0.2652 - accuracy=0.9006 - val_loss=0.3362 - val_accuracy=0.8969 - tiempo=3.15s


Epoch 15/15 - loss=0.3451 - accuracy=0.8733 - val_loss=0.4152 - val_accuracy=0.8594 - tiempo=3.04s


MLP Base accuracy: 0.7933
CNN Base accuracy: 0.85


# 6. Regularizar

In [6]:

def build_cnn_regularized(input_shape, num_classes, lr=1e-3, dropout_rate=0.4):
    inputs = keras.layers.Input(shape=input_shape)
    x = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D()(x)

    x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D()(x)

    x = keras.layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D()(x)

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(dropout_rate)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="CNN_Regularizada")
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
    return model

regularized_configs = [
    {"name": "dropout_0.2_lr_1e-3", "dropout": 0.2, "lr": 1e-3},
    {"name": "dropout_0.4_lr_1e-3", "dropout": 0.4, "lr": 1e-3},
    {"name": "dropout_0.2_lr_5e-4", "dropout": 0.2, "lr": 5e-4},
    {"name": "dropout_0.4_lr_5e-4", "dropout": 0.4, "lr": 5e-4},
]

regularized_results = {}
for cfg_idx, cfg in enumerate(regularized_configs):
    model = build_cnn_regularized(input_shape, num_classes, cfg["lr"], cfg["dropout"])
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_weights = None
    started = time.time()

    for epoch in range(1, 16):
        epoch_started = time.time()
        rng = np.random.default_rng(SEED + 300 + cfg_idx * 100 + epoch)
        indices = rng.permutation(len(X_train))
        train_losses, train_accs = [], []

        for start_i in range(0, len(indices), batch_size):
            batch_idx = indices[start_i:start_i + batch_size]
            batch_x = augment_batch(X_train[batch_idx], rng)
            metrics = model.train_on_batch(batch_x, y_train[batch_idx], return_dict=True)
            train_losses.append(float(metrics["loss"]))
            train_accs.append(float(metrics["accuracy"]))

        val_losses, val_accs = [], []
        for start_i in range(0, len(X_val), batch_size):
            metrics = model.test_on_batch(
                X_val[start_i:start_i + batch_size], y_val[start_i:start_i + batch_size], return_dict=True
            )
            val_losses.append(float(metrics["loss"]))
            val_accs.append(float(metrics["accuracy"]))

        epoch_values = {
            "loss": float(np.mean(train_losses)), "accuracy": float(np.mean(train_accs)),
            "val_loss": float(np.mean(val_losses)), "val_accuracy": float(np.mean(val_accs)),
        }
        for key, value in epoch_values.items():
            history[key].append(value)

        if epoch_values["val_loss"] < best_val_loss:
            best_val_loss = epoch_values["val_loss"]
            best_weights = model.get_weights()

        print(f"{cfg['name']} | Epoch {epoch:02d}/15 - loss={epoch_values['loss']:.4f} - accuracy={epoch_values['accuracy']:.4f} - val_loss={epoch_values['val_loss']:.4f} - val_accuracy={epoch_values['val_accuracy']:.4f} - tiempo={time.time()-epoch_started:.2f}s")

    model.set_weights(best_weights)
    test_acc, precision, recall, f1, preds, probs = evaluate_and_predict(model)
    regularized_results[cfg["name"]] = {
        "model": model, "history": history, "test_accuracy": test_acc,
        "precision_macro": precision, "recall_macro": recall, "f1_macro": f1,
        "predictions": preds, "probabilities": probs,
        "val_loss_min": best_val_loss, "time": time.time() - started,
        "config": cfg,
    }
    print(f"{cfg['name']} final: mejor val_loss={best_val_loss:.4f}, test_accuracy={test_acc:.4f}\n")

best_reg_name = min(regularized_results, key=lambda name: (
    regularized_results[name]["val_loss_min"]
))
best_reg = regularized_results[best_reg_name]
best_cnn_regularized = best_reg["model"]
print("Mejor CNN regularizada:", best_reg_name, best_reg["config"])

best_history = best_reg["history"]
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(best_history["loss"], label="train")
plt.plot(best_history["val_loss"], label="validation")
plt.title("CNN regularizada: pérdida")
plt.xlabel("Época")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(best_history["accuracy"], label="train")
plt.plot(best_history["val_accuracy"], label="validation")
plt.title("CNN regularizada: accuracy")
plt.xlabel("Época")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "curvas_cnn_regularizada.png"), dpi=150, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, best_reg["predictions"]), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Matriz de confusión — CNN regularizada")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "cm_cnn_regularizada.png"), dpi=150, bbox_inches="tight")
plt.close()

best_cnn_regularized.save(os.path.join(MODELS_DIR, "modelo_cnn_final_ep3.keras"))

comparison_rows = [
    {
        "Modelo": "MLP Base",
        "Test Accuracy": mlp_test[0], "Precision Macro": mlp_test[1],
        "Recall Macro": mlp_test[2], "F1-Score Macro": mlp_test[3],
        "Parámetros Totales": int(mlp_base.count_params()),
        "Tiempo Entrenamiento (s)": mlp_time,
    },
    {
        "Modelo": "CNN Base",
        "Test Accuracy": cnn_test[0], "Precision Macro": cnn_test[1],
        "Recall Macro": cnn_test[2], "F1-Score Macro": cnn_test[3],
        "Parámetros Totales": int(cnn_base.count_params()),
        "Tiempo Entrenamiento (s)": cnn_time,
    },
]
for name, result in regularized_results.items():
    comparison_rows.append({
        "Modelo": f"CNN Regularizada - {name}",
        "Test Accuracy": result["test_accuracy"], "Precision Macro": result["precision_macro"],
        "Recall Macro": result["recall_macro"], "F1-Score Macro": result["f1_macro"],
        "Parámetros Totales": int(result["model"].count_params()),
        "Tiempo Entrenamiento (s)": result["time"],
    })

metrics_df = pd.DataFrame(comparison_rows)
metrics_df.to_csv(os.path.join(IMAGES_DIR, "metricas_comparacion_ep3.csv"), index=False)
formatted = metrics_df.copy()
for col in ["Test Accuracy", "Precision Macro", "Recall Macro", "F1-Score Macro"]:
    formatted[col] = formatted[col].map(lambda value: f"{value:.4f}")
formatted["Parámetros Totales"] = formatted["Parámetros Totales"].map(lambda value: f"{int(value):,}")
formatted["Tiempo Entrenamiento (s)"] = formatted["Tiempo Entrenamiento (s)"].map(lambda value: f"{value:.2f}")
display(formatted)

conv_layers = [layer for layer in best_cnn_regularized.layers if isinstance(layer, keras.layers.Conv2D)]
last_conv_name = conv_layers[-1].name
grad_model = keras.Model(
    inputs=best_cnn_regularized.input,
    outputs=[best_cnn_regularized.get_layer(last_conv_name).output, best_cnn_regularized.output]
)

predictions = best_reg["predictions"]
correct_idx = np.where(predictions == y_test)[0]
incorrect_idx = np.where(predictions != y_test)[0]
selected = []
if len(correct_idx):
    selected.extend(correct_idx[np.linspace(0, len(correct_idx) - 1, min(3, len(correct_idx)), dtype=int)].tolist())
if len(incorrect_idx):
    selected.extend(incorrect_idx[np.linspace(0, len(incorrect_idx) - 1, min(3, len(incorrect_idx)), dtype=int)].tolist())
if len(selected) < 6:
    remaining = [i for i in range(len(X_test)) if i not in selected]
    rng = np.random.default_rng(SEED)
    rng.shuffle(remaining)
    selected.extend(remaining[:6 - len(selected)])
selected = selected[:6]

plt.figure(figsize=(15, 10))
for plot_idx, test_idx in enumerate(selected):
    image = X_test[test_idx]
    image_tensor = tf.convert_to_tensor(image[np.newaxis, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        conv_output, probabilities = grad_model(image_tensor, training=False)
        pred_idx = tf.argmax(probabilities[0])
        score = probabilities[0, pred_idx]
    gradients = tape.gradient(score, conv_output)
    pooled = tf.reduce_mean(gradients, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_output[0] * pooled, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    if tf.reduce_max(heatmap) > 0:
        heatmap = heatmap / tf.reduce_max(heatmap)
    heatmap = tf.image.resize(heatmap[..., None], (128, 128)).numpy()[..., 0]

    real_idx = int(y_test[test_idx])
    pred_idx = int(pred_idx.numpy())
    confidence = float(probabilities[0, pred_idx].numpy())
    status = "ACIERTO" if pred_idx == real_idx else "ERROR"

    plt.subplot(2, 3, plot_idx + 1)
    plt.imshow(image)
    plt.imshow(heatmap, cmap="jet", alpha=0.45)
    plt.title(f"{status}: Real {class_names[real_idx]}\nPred: {class_names[pred_idx]} ({confidence:.1%})")
    plt.axis("off")

plt.suptitle("Auditoría visual Grad-CAM — CNN regularizada", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "gradcam_auditoria_ep3.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Grad-CAM guardado en:", os.path.join(IMAGES_DIR, "gradcam_auditoria_ep3.png"))

dropout_0.2_lr_1e-3 | Epoch 01/15 - loss=0.5853 - accuracy=0.7865 - val_loss=1.5237 - val_accuracy=0.3146 - tiempo=11.05s


dropout_0.2_lr_1e-3 | Epoch 02/15 - loss=0.2744 - accuracy=0.9081 - val_loss=2.0997 - val_accuracy=0.2542 - tiempo=9.52s


dropout_0.2_lr_1e-3 | Epoch 03/15 - loss=0.1981 - accuracy=0.9347 - val_loss=2.3724 - val_accuracy=0.2365 - tiempo=9.39s


dropout_0.2_lr_1e-3 | Epoch 04/15 - loss=0.1749 - accuracy=0.9399 - val_loss=3.5343 - val_accuracy=0.2365 - tiempo=9.32s


dropout_0.2_lr_1e-3 | Epoch 05/15 - loss=0.1669 - accuracy=0.9432 - val_loss=4.9024 - val_accuracy=0.2458 - tiempo=9.49s


dropout_0.2_lr_1e-3 | Epoch 06/15 - loss=0.1429 - accuracy=0.9510 - val_loss=3.7083 - val_accuracy=0.2115 - tiempo=9.42s


... (56 outputs left out)

dropout_0.4_lr_5e-4 | Epoch 15/15 - loss=0.0966 - accuracy=0.9697 - val_loss=0.3606 - val_accuracy=0.8604 - tiempo=18.16s


dropout_0.4_lr_5e-4 final: mejor val_loss=0.3606, test_accuracy=0.8367

Mejor CNN regularizada: dropout_0.2_lr_5e-4 {'name': 'dropout_0.2_lr_5e-4', 'dropout': 0.2, 'lr': 0.0005}


,Modelo,Test Accuracy,Precision Macro,Recall Macro,F1-Score Macro,Parámetros Totales,Tiempo Entrenamiento (s)
0,MLP Base,0.7933,0.8105,0.7933,0.7959,"25,298,692",69.82
1,CNN Base,0.8500,0.8659,0.8500,0.8487,"110,276",47.55
2,CNN Regularizada - dropout_0.2_lr_1e-3,0.6133,0.8333,0.6133,0.5754,"111,172",139.44
3,CNN Regularizada - dropout_0.4_lr_1e-3,0.8767,0.8986,0.8767,0.8785,"111,172",250.31
4,CNN Regularizada - dropout_0.2_lr_5e-4,0.9667,0.9698,0.9667,0.9671,"111,172",262.71
5,CNN Regularizada - dropout_0.4_lr_5e-4,0.8367,0.8892,0.8367,0.8331,"111,172",267.84


Grad-CAM guardado en: C:\Users\sj-alumno\Downloads\PlantVillageTomato\images\gradcam_auditoria_ep3.png
